# File Checkpointing: Rewinding Your Agent's File Changes

**Solution notebook.** Every cell is filled in.

When an agent edits your files, you sometimes want those edits gone. A forked session branches the *conversation*, but it does not give you your files back. **File checkpointing** closes that gap. The SDK backs up a file before the agent modifies it through the file-editing tools, so you can restore it later.

In this notebook you will:

1. Turn checkpointing on, and learn why a second option is needed to receive checkpoint IDs.
2. Capture a checkpoint ID from the message stream, and see why it is not the same thing as a session ID.
3. Rewind: restore a file the agent rewrote and delete a file the agent created.
4. Prove each result with filesystem ground truth (hashes and file existence), never by trusting the agent's own words.
5. Discover what checkpointing does *not* undo, including a change made through the Bash tool.

Because the agent decides how to do the work, its wording and its tool choices vary from run to run. Each demo tells you what to look for instead of expecting identical text.

### Cell map

| Cell | What it does |
|---|---|
| 1 | Install the SDK |
| 2 | Configure the API key |
| 3 | Choose the model |
| 4 | Create the sample files and record their hashes |
| 5 | Verify the folder tree and file contents |
| 6 | Define the prompts as variables |
| 7 | Define the checkpointing options |
| 8 | Run the agent and capture the checkpoint ID and session ID |
| 9 | Show what the agent changed |
| 10 | Rewind the files |
| 11 | Ground truth: files are back |
| 12 | Conversation is not rewound |
| 13 | Demo 2: the Bash blind spot (run) |
| 14 | Demo 2: rewind and ground truth |
| 15 | Summary of what rewinds and what does not |

## Cell 1: Install the Claude Agent SDK

This cell installs the **Claude Agent SDK** for Python, the only package this notebook needs. The package bundles the Claude Code CLI that actually runs the agent loop, so there is nothing else to install.

| Part | Meaning |
|---|---|
| `!pip install` | The leading `!` runs a shell command from a notebook cell. |
| `claude-agent-sdk==0.2.167` | The package, pinned to one exact version so every result matches the course. |
| `-q` | Quiet mode, to keep the output short. |

File checkpointing needs a reasonably recent SDK. If a later cell reports that `enable_file_checkpointing` or `rewind_files` is not recognised, your installed version is too old, and upgrading fixes it.

If the same version is already installed in this Colab session, pip reports that the requirement is satisfied and finishes quickly. If a different version is installed, pip replaces it with the pinned one.

Run the cell and let it finish before you continue.

In [1]:
# Cell 1: Install the Claude Agent SDK
# We pin the Claude Agent SDK to a specific version so every example runs
# exactly as shown in the course. To use the latest version, run:
# pip install -U claude-agent-sdk
# Or substitute your preferred version below. Newer versions may behave
# differently from the videos.
!pip install claude-agent-sdk==0.2.167 -q

## Cell 2: Configure your API key

The agent calls the Claude API, so the SDK needs your Anthropic API key. In Colab the safe way to provide it is **Colab Secrets**, which keeps the key out of the notebook itself.

**One-time setup in Colab**

1. Create a key in the Claude Console at https://platform.claude.com/ under **API Keys**, and copy it.
2. In Colab, click the **key icon** in the left sidebar.
3. Click **Add new secret**.
4. Name it exactly `ANTHROPIC_API_KEY` and paste your key as the value.
5. Switch **Notebook access** on for this notebook.

The cell reads the secret with `userdata.get("ANTHROPIC_API_KEY")` and stores it in the environment variable `ANTHROPIC_API_KEY`, which is where the SDK looks for it.

**Running outside Colab?** Set the `ANTHROPIC_API_KEY` environment variable in your terminal instead, and skip this cell.

In [2]:
# Cell 2: Configure the API key from Colab Secrets
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Cell 3: Choose the model

Every agent in this notebook gets its model from one variable, `MODEL_NAME`. Changing this single line updates the model used across the entire notebook.

This notebook uses Claude Haiku 4.5 (`claude-haiku-4-5`), which is fast and more than capable for the small file tasks in these demos. To try a different model, look up the current model names on the models page linked in the code comment and replace the string.

Run the cell. It prints nothing.

In [3]:
# Cell 3: Choose the model
# See latest models at: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-4-5"

## Cell 4: Create the sample files and record their hashes

Checkpointing is only convincing if you can prove what happened on disk. So this cell builds a small playground, a seed library for a community swap day, and records a **fingerprint** of each file before any agent touches it.

| File | Contents | Purpose |
|---|---|---|
| `catalog.txt` | Six seed varieties with packet counts, in unsorted order | Demo 1: the agent rewrites this file |
| `notes.txt` | Three short lines about swap-day logistics | Demo 2: the agent appends a line to this file |

The fingerprint is a **SHA-256 hash**, computed by the helper `sha256_of()`. Two files with identical bytes always have the same hash, and changing even one character changes it completely. That gives you a yes or no answer to "is this file back to its original state?" without reading anything by eye.

The cell also sets up paths for `summary.txt` and `parking.txt`, two files the agent will create later. They do not exist yet.

The cell starts by deleting and recreating the folder, so **re-running Cell 4 resets the whole playground** to its original state. Run it now.

In [4]:
# Cell 4: Create the sample files and record their SHA-256 hashes
import hashlib
import shutil
from pathlib import Path

DEMO_DIR = Path("/content/seedlibrary_demo")

# Start from a clean folder so re-running this cell always resets the demo.
if DEMO_DIR.exists():
    shutil.rmtree(DEMO_DIR)
DEMO_DIR.mkdir(parents=True)

CATALOG_PATH = DEMO_DIR / "catalog.txt"
NOTES_PATH = DEMO_DIR / "notes.txt"
SUMMARY_PATH = DEMO_DIR / "summary.txt"   # created by the agent in Demo 1
PARKING_PATH = DEMO_DIR / "parking.txt"   # created by the agent in Demo 2

CATALOG_LINES = [
    "marigold: 40 packets",
    "basil: 12 packets",
    "okra: 75 packets",
    "zinnia: 28 packets",
    "dill: 9 packets",
    "cosmos: 51 packets",
]
NOTES_LINES = [
    "Swap day is the first Saturday of the month.",
    "Tables open for setup at 8 AM and doors open at 9 AM.",
    "Bring labelled envelopes for any seeds you want to share.",
]

CATALOG_PATH.write_text("\n".join(CATALOG_LINES) + "\n")
NOTES_PATH.write_text("\n".join(NOTES_LINES) + "\n")


def sha256_of(path):
    """Return the SHA-256 fingerprint of a file's bytes."""
    return hashlib.sha256(path.read_bytes()).hexdigest()


ORIGINAL_HASHES = {
    "catalog.txt": sha256_of(CATALOG_PATH),
    "notes.txt": sha256_of(NOTES_PATH),
}

for name, digest in ORIGINAL_HASHES.items():
    print(f"{name}: {digest[:16]}...")

catalog.txt: 6658974d01ed8222...
notes.txt: 4a78ef5fa376f44f...


## Cell 5: Verify the folder tree and file contents

Before any agent runs, confirm the starting state with your own eyes. This cell prints the folder tree, then each file's hash prefix and full contents.

What to look for:

- Exactly two files in the folder: `catalog.txt` and `notes.txt`.
- The six catalog lines in the unsorted order you saw in Cell 4.
- No `summary.txt` and no `parking.txt` yet.

Every demo in this notebook is measured against this starting state, which is why verification comes before the agent does anything.

In [5]:
# Cell 5: Verify the folder tree and file contents
print("Folder tree:")
print(DEMO_DIR)
for path in sorted(DEMO_DIR.rglob("*")):
    print("  ", path.relative_to(DEMO_DIR))

print()
for path in sorted(DEMO_DIR.glob("*.txt")):
    print(f"--- {path.name} (sha256 {sha256_of(path)[:12]}...) ---")
    print(path.read_text())

Folder tree:
/content/seedlibrary_demo
   catalog.txt
   notes.txt

--- catalog.txt (sha256 6658974d01ed...) ---
marigold: 40 packets
basil: 12 packets
okra: 75 packets
zinnia: 28 packets
dill: 9 packets
cosmos: 51 packets

--- notes.txt (sha256 4a78ef5fa376...) ---
Swap day is the first Saturday of the month.
Tables open for setup at 8 AM and doors open at 9 AM.
Bring labelled envelopes for any seeds you want to share.



## Cell 6: Define the prompts as variables

All three prompts live in one cell so the long strings stay out of the cells where you will focus on the SDK options.

| Variable | Used in | What it asks the agent to do |
|---|---|---|
| `DEMO1_PROMPT` | Cell 8 | Sort `catalog.txt` by packet count, highest first, and create `summary.txt` listing the top three |
| `FOLLOW_UP_PROMPT` | Cell 12 | Describe the changes it made earlier, from memory, with no tools |
| `DEMO2_PROMPT` | Cell 13 | Append a line to `notes.txt` with the Bash tool and create `parking.txt` with the Write tool |

`DEMO1_PROMPT` is a **goal prompt**: it states the outcome and leaves the choice of tools to the agent. `DEMO2_PROMPT` is a deliberate exception. It names the tools because the entire point of Demo 2 is that *which tool* made a change decides whether checkpointing can undo it. Cell 13 explains this again.

Run the cell. It prints nothing.

In [6]:
# Cell 6: Define the prompts as variables
DEMO1_PROMPT = (
    "Take the seed catalog in /content/seedlibrary_demo/catalog.txt and rewrite "
    "that same file so the varieties are sorted by packet count, highest first. "
    "Then create a new file called summary.txt in the same folder that lists "
    "the three varieties with the most packets."
)

FOLLOW_UP_PROMPT = (
    "Describe the changes you made earlier in this conversation. "
    "Answer from memory only and do not use any tools."
)

PARKING_LINE = "Parking note: volunteers use the east gate."
PARKING_FILE_TEXT = "The east gate is next to the community hall."

DEMO2_PROMPT = (
    "Do two things. First, use the Bash tool to append this exact line to the "
    f"end of /content/seedlibrary_demo/notes.txt: {PARKING_LINE} "
    "Second, use the Write tool to create /content/seedlibrary_demo/parking.txt "
    f"containing this single line: {PARKING_FILE_TEXT}"
)

## Cell 7: Define the checkpointing options

This is the first new SDK concept. Checkpointing is off by default, so you turn it on through `ClaudeAgentOptions`.

| Option | Value | What it does |
|---|---|---|
| `enable_file_checkpointing` | `True` | Tells the SDK to back up a file before the agent modifies it through the file-editing tools, so it can be restored later. |
| `extra_args` | `{"replay-user-messages": None}` | Passes an extra flag to the underlying Claude Code CLI. Without it, user messages in the stream arrive without a UUID. Those UUIDs *are* the checkpoint IDs, so this option is required. A value of `None` means a flag with no argument. |
| `allowed_tools` | `["Write", "Edit"]` | Pre-approves exactly the two file-editing tools this demo needs. Nothing else is granted, which keeps the demo least-privilege. |
| `model` | `MODEL_NAME` | The model from Cell 3. |
| `cwd` | `"/content"` | The working directory for the agent. |

Two design notes:

- **Permissions.** The agent also needs to read `catalog.txt` before rewriting it. Reading files inside the working directory is approved automatically, so `Read` does not need to be listed. We use `allowed_tools` rather than a broader permission mode so you can see exactly which tools are approved.
- **Which tools are tracked.** Checkpointing tracks changes made through the `Write`, `Edit`, and `NotebookEdit` tools. This demo allows `Write` and `Edit`, and either one counts as tracked, so it does not matter which one the agent picks for `catalog.txt`.

Run the cell. It prints the two checkpointing settings so you can confirm they took effect.

In [7]:
# Cell 7: Define the checkpointing options
from claude_agent_sdk import ClaudeAgentOptions

checkpoint_options = ClaudeAgentOptions(
    enable_file_checkpointing=True,
    extra_args={"replay-user-messages": None},
    allowed_tools=["Write", "Edit"],
    model=MODEL_NAME,
    cwd="/content",
)

print("File checkpointing on:", checkpoint_options.enable_file_checkpointing)
print("extra_args:", checkpoint_options.extra_args)

File checkpointing on: True
extra_args: {'replay-user-messages': None}


## Cell 8: Run the agent and capture the checkpoint ID and session ID

This cell runs Demo 1 and captures the two identifiers you will need to rewind. They look similar (both are UUID strings), but they are completely different things.

| | Checkpoint ID | Session ID |
|---|---|---|
| **What it identifies** | A point in time inside the session: a *user message* | The whole session |
| **Where it comes from** | `UserMessage.uuid` in the stream | `ResultMessage.session_id` at the end of the stream |
| **What it is for** | Tells `rewind_files()` *which state to restore* | Tells `resume` *which session to reopen* |
| **Variable here** | `CHECKPOINT_ID` | `SESSION_ID` |

**The checkpoint ID comes from a `UserMessage`, not from the session.** This is the most common point of confusion. You need both: the session ID to reopen the session later, and the checkpoint ID to say which moment to restore.

How the cell works:

1. `ClaudeSDKClient` is used as an async context manager. It opens a connection to the agent and closes it when the block ends. Colab allows `async with` and `async for` directly at the top level of a cell.
2. `client.query(...)` sends the prompt. `client.receive_response()` then yields every message until the final `ResultMessage`.
3. The first `UserMessage` that has a `uuid` is kept as `CHECKPOINT_ID`. The documentation says that rewinding to the first user message restores tracked files to their original state.
4. The `ResultMessage` carries the `session_id`, and its `result` field holds the agent's final answer.
5. A counter shows how many `UserMessage` objects with a UUID arrived. This is for curiosity only. All proof in this notebook comes from the filesystem.

What to look for: a printed `CHECKPOINT_ID` and `SESSION_ID`, both real UUIDs, and a short summary from the agent. If `CHECKPOINT_ID` prints `None`, the `extra_args` option from Cell 7 was not applied.

In [8]:
# Cell 8: Run the agent and capture the checkpoint ID and session ID
from claude_agent_sdk import ClaudeSDKClient, UserMessage, ResultMessage

CHECKPOINT_ID = None
SESSION_ID = None
user_message_count = 0

async with ClaudeSDKClient(options=checkpoint_options) as client:
    await client.query(DEMO1_PROMPT)

    async for message in client.receive_response():
        if isinstance(message, UserMessage) and message.uuid:
            user_message_count += 1
            if CHECKPOINT_ID is None:
                CHECKPOINT_ID = message.uuid
        if isinstance(message, ResultMessage):
            SESSION_ID = message.session_id
            if message.subtype == "success":
                print(message.result)

print()
print("UserMessage objects with a uuid:", user_message_count)
print("CHECKPOINT_ID:", CHECKPOINT_ID)
print("SESSION_ID:", SESSION_ID)

Done! I've completed both tasks:

1. **Rewritten catalog.txt** - The varieties are now sorted by packet count from highest to lowest:
   - okra: 75 packets
   - cosmos: 51 packets
   - marigold: 40 packets
   - zinnia: 28 packets
   - basil: 12 packets
   - dill: 9 packets

2. **Created summary.txt** - A new file listing the top 3 varieties by packet count in the same folder.

UserMessage objects with a uuid: 4
CHECKPOINT_ID: ec84984d-fa2d-47dd-a6fd-de8b370150af
SESSION_ID: 24442334-751f-4c49-8077-e69d492e1b70


## Cell 9: Show the changes

Now check what the agent actually did, using the filesystem rather than the agent's own report.

The cell compares the current hash of `catalog.txt` with the original from Cell 4, checks whether `summary.txt` exists, and prints both files.

What to look for:

- `catalog.txt` hash differs from the original: `True`.
- `summary.txt` exists: `True`.
- The catalog lines are now sorted by packet count with the largest first, and the summary lists the three largest varieties.

The exact wording of `summary.txt` will vary between runs. That is normal for a live agent. What matters is that the files changed, so there is something to rewind.

In [9]:
# Cell 9: Show the changes
print("catalog.txt hash differs from original:",
      sha256_of(CATALOG_PATH) != ORIGINAL_HASHES["catalog.txt"])
print("summary.txt exists:", SUMMARY_PATH.exists())

print()
print("--- catalog.txt now ---")
print(CATALOG_PATH.read_text())

if SUMMARY_PATH.exists():
    print("--- summary.txt ---")
    print(SUMMARY_PATH.read_text())

catalog.txt hash differs from original: True
summary.txt exists: True

--- catalog.txt now ---
okra: 75 packets
cosmos: 51 packets
marigold: 40 packets
zinnia: 28 packets
basil: 12 packets
dill: 9 packets

--- summary.txt ---
Top 3 Varieties by Packet Count:

1. okra: 75 packets
2. cosmos: 51 packets
3. marigold: 40 packets



## Cell 10: Rewind the files

This is the core of the lesson. The agent's session has finished, and you now ask the SDK to put the files back.

The pattern has three parts, and each one exists for a reason.

| Step | Code | Why |
|---|---|---|
| Resume the session | `ClaudeAgentOptions(enable_file_checkpointing=True, resume=SESSION_ID, ...)` | The checkpoints belong to the session that created them, so you reopen that session. Checkpointing must be enabled on the resumed session too. |
| Open the connection with an empty prompt | `await client.query("")` | `rewind_files()` talks to a live connection. The empty prompt opens that connection without asking the agent to do any new work. |
| Rewind inside the loop, then stop | `await client.rewind_files(CHECKPOINT_ID)` then `break` | Once the response loop has finished, the connection is closed. The documentation lists calling `rewind_files()` after the loop as a cause of a transport error, so the call goes *inside* the loop, and `break` ends the loop right after it. |

What `rewind_files()` does: it restores files the agent *modified* to their content at the checkpoint, and it deletes files the agent *created* after it. Notice that the resumed options do not need `allowed_tools` or `extra_args`. This session is not asking the agent to do anything, only to restore.

The cell prints a line when the call returns without raising an error. That line is **not** proof that the files are back. Proof comes from the filesystem in the next cell.

In [10]:
# Cell 10: Rewind the files
rewind_options = ClaudeAgentOptions(
    enable_file_checkpointing=True,
    resume=SESSION_ID,
    model=MODEL_NAME,
    cwd="/content",
)

async with ClaudeSDKClient(options=rewind_options) as client:
    await client.query("")
    async for message in client.receive_response():
        await client.rewind_files(CHECKPOINT_ID)
        break

print("rewind_files() returned without an error for checkpoint:", CHECKPOINT_ID)

rewind_files() returned without an error for checkpoint: ec84984d-fa2d-47dd-a6fd-de8b370150af


## Cell 11: Ground truth, the files are back

Now prove the rewind worked, using the same two kinds of evidence as before: a hash comparison and a file-existence check.

What to look for:

- `catalog.txt` matches the original: `True`. The file is byte-for-byte what you created in Cell 4.
- `summary.txt` exists: `False`. A file the agent created has been deleted.
- The catalog printed below is back in its original unsorted order.

If either check disagrees, the rewind did not do what the documentation describes. Compare your result with the checks in Cell 9 and note which one differs.

In [11]:
# Cell 11: Ground truth, the files are back
print("catalog.txt matches original:",
      sha256_of(CATALOG_PATH) == ORIGINAL_HASHES["catalog.txt"])
print("summary.txt exists:", SUMMARY_PATH.exists())

print()
print("--- catalog.txt now ---")
print(CATALOG_PATH.read_text())

catalog.txt matches original: True
summary.txt exists: False

--- catalog.txt now ---
marigold: 40 packets
basil: 12 packets
okra: 75 packets
zinnia: 28 packets
dill: 9 packets
cosmos: 51 packets



## Cell 12: The conversation is not rewound

The files went back. Now ask whether the *agent* went back with them.

Rewinding restores files on disk and nothing else. The conversation history stays exactly as it was, so the agent should still remember sorting the catalog and writing the summary, even though neither change exists on disk any more.

This cell resumes the original session with a plain `query()` call, using `resume=SESSION_ID`, and asks the agent to describe its earlier changes. Two details keep the test clean:

- No `allowed_tools` are passed, and the prompt tells the agent to answer from memory without using tools. The agent therefore cannot cheat by re-reading the files, which now show the original state.
- The result is read from the final `ResultMessage`, as in earlier notebooks.

What to look for: the agent describes the sort and the summary file as things it did. The wording will vary. The point is that its *memory* includes changes that your *filesystem* no longer does.

The files go back. The memory does not.

In [12]:
# Cell 12: The conversation is not rewound
from claude_agent_sdk import query

async for message in query(
    prompt=FOLLOW_UP_PROMPT,
    options=ClaudeAgentOptions(
        resume=SESSION_ID,
        model=MODEL_NAME,
        cwd="/content",
    ),
):
    if isinstance(message, ResultMessage) and message.subtype == "success":
        print(message.result)

Based on my memory of this conversation, I made the following changes:

1. **Sorted the catalog.txt file** - I reordered the seed varieties in `/content/seedlibrary_demo/catalog.txt` by packet count from highest to lowest:
   - okra (75 packets) moved to first position
   - cosmos (51 packets) moved to second
   - marigold (40 packets) moved to third
   - zinnia (28 packets) moved to fourth
   - basil (12 packets) moved to fifth
   - dill (9 packets) stayed at bottom

2. **Created summary.txt** - I created a new file called `summary.txt` in the same folder that listed the top 3 varieties with the most packets:
   - okra: 75 packets
   - cosmos: 51 packets
   - marigold: 40 packets

The original file had the varieties in a seemingly random order, and I reorganized it to display them in descending order by packet count, then extracted the top 3 into a summary file.


## Cell 13: Demo 2, the Bash blind spot (run)

Checkpointing has a limit that catches many people out. It only tracks changes made through the file-editing tools: `Write`, `Edit`, and `NotebookEdit`. A change made by running a shell command through the `Bash` tool is **not** tracked.

To show this, one session does one of each:

| Change | Tool | Tracked? (per the documentation) |
|---|---|---|
| Append a parking note to `notes.txt` | `Bash` | No |
| Create `parking.txt` | `Write` | Yes |

**A deliberate exception to the goal-prompt rule.** Elsewhere in this course you give the agent a goal and let it choose tools. Here `DEMO2_PROMPT` names `Bash` and `Write` on purpose, because the lesson is about the *type of tool*. If the agent chose its own tools, the demo would not reliably isolate that variable.

The options are fresh and use `allowed_tools=["Bash", "Write"]`, so both calls are pre-approved. This is a new session, so it gets its own `DEMO2_CHECKPOINT_ID` and `DEMO2_SESSION_ID`, captured exactly as in Cell 8. Checkpoints belong to the session that created them, so Demo 1's IDs cannot be used here.

What to look for after the run:

- `parking.txt` now exists.
- `notes.txt` has a fourth line, the parking note.
- The `notes.txt` hash no longer matches the original.

Agents do not always follow tool instructions exactly. If the output shows the agent used a different tool than you asked for, note it, because it changes what the next cell should show.

In [13]:
# Cell 13: Demo 2, the Bash blind spot (run)

# DEMO2_PROMPT = (
#     "Do two things. First, use the Bash tool to append this exact line to the "
#     f"end of /content/seedlibrary_demo/notes.txt: {PARKING_LINE} "
#     "Second, use the Write tool to create /content/seedlibrary_demo/parking.txt "
#     f"containing this single line: {PARKING_FILE_TEXT}"
# )

demo2_options = ClaudeAgentOptions(
    enable_file_checkpointing=True,
    extra_args={"replay-user-messages": None},
    allowed_tools=["Bash", "Write"],
    model=MODEL_NAME,
    cwd="/content",
)

DEMO2_CHECKPOINT_ID = None
DEMO2_SESSION_ID = None

print("Before the run, parking.txt exists:", PARKING_PATH.exists())

async with ClaudeSDKClient(options=demo2_options) as client:
    await client.query(DEMO2_PROMPT)

    async for message in client.receive_response():
        if isinstance(message, UserMessage) and message.uuid and DEMO2_CHECKPOINT_ID is None:
            DEMO2_CHECKPOINT_ID = message.uuid
        if isinstance(message, ResultMessage):
            DEMO2_SESSION_ID = message.session_id

print()
print("DEMO2_CHECKPOINT_ID:", DEMO2_CHECKPOINT_ID)
print("DEMO2_SESSION_ID:", DEMO2_SESSION_ID)

print()
print("After the run, parking.txt exists:", PARKING_PATH.exists())
print("notes.txt matches original:",
      sha256_of(NOTES_PATH) == ORIGINAL_HASHES["notes.txt"])

print()
print("--- notes.txt ---")
print(NOTES_PATH.read_text())

if PARKING_PATH.exists():
    print("--- parking.txt ---")
    print(PARKING_PATH.read_text())

Before the run, parking.txt exists: False

DEMO2_CHECKPOINT_ID: ba70a0f7-3fc0-457a-8e4c-01ff8eca5535
DEMO2_SESSION_ID: 76f933c3-c466-463c-acf4-136e9966a258

After the run, parking.txt exists: True
notes.txt matches original: False

--- notes.txt ---
Swap day is the first Saturday of the month.
Tables open for setup at 8 AM and doors open at 9 AM.
Bring labelled envelopes for any seeds you want to share.
Parking note: volunteers use the east gate.

--- parking.txt ---
The east gate is next to the community hall.


## Cell 14: Demo 2, rewind and ground truth

Now rewind Demo 2's session to its checkpoint, using exactly the pattern from Cell 10 with the Demo 2 identifiers, and then check both files.

Two checks matter, and they are meant to disagree:

| Check | Expected result | Why |
|---|---|---|
| `parking.txt` exists | `False` | It was created through `Write`, which is tracked, so the rewind deletes it. |
| Parking note is still in `notes.txt` | `True` | It was appended through `Bash`, which is not tracked, so the rewind never touched it. |

The cell also prints `notes.txt` in full so you can see the surviving line yourself.

**This is the lesson people trip over.** Both changes happened in the same session, yet only one was undone. Checkpointing is not a general-purpose undo. It only covers what the editing tools touched. Anything that changes files through a shell command, or through any route that is not `Write`, `Edit`, or `NotebookEdit`, is outside its reach.

In [14]:
# Cell 14: Demo 2, rewind and ground truth
rewind_options_2 = ClaudeAgentOptions(
    enable_file_checkpointing=True,
    resume=DEMO2_SESSION_ID,
    model=MODEL_NAME,
    cwd="/content",
)

async with ClaudeSDKClient(options=rewind_options_2) as client:
    await client.query("")
    async for message in client.receive_response():
        await client.rewind_files(DEMO2_CHECKPOINT_ID)
        break

write_change_undone = not PARKING_PATH.exists()
bash_change_survived = PARKING_LINE in NOTES_PATH.read_text()

print("parking.txt exists after rewind:", PARKING_PATH.exists())
print("Parking note still in notes.txt:", bash_change_survived)
print("notes.txt matches original:",
      sha256_of(NOTES_PATH) == ORIGINAL_HASHES["notes.txt"])

print()
print("Write change undone:", write_change_undone)
print("Bash change survived:", bash_change_survived)

print()
print("--- notes.txt ---")
print(NOTES_PATH.read_text())

parking.txt exists after rewind: False
Parking note still in notes.txt: True
notes.txt matches original: False

Write change undone: True
Bash change survived: True

--- notes.txt ---
Swap day is the first Saturday of the month.
Tables open for setup at 8 AM and doors open at 9 AM.
Bring labelled envelopes for any seeds you want to share.
Parking note: volunteers use the east gate.



## Cell 15: Summary, what rewinds and what does not

Everything you saw in this notebook, in one table.

| What happened | Undone by `rewind_files()`? | Why |
|---|---|---|
| A file **created** through `Write` | Yes, it is deleted | Tracked: the SDK backs up files before the editing tools change them |
| A file **modified** through `Write`, `Edit` or `NotebookEdit` | Yes, it is restored | Tracked |
| A file changed through the **Bash** tool | No | Shell commands are not tracked |
| Edits made by a **subagent** | No | Subagent edits are not tracked |
| A checkpoint from a **different session** | No | Checkpoints belong to the session that created them |
| **Directory** changes (create, move, delete) | No | Only file content is restored |
| The agent's **conversation memory** | No | Rewinding restores files only. The conversation is untouched. |

Remember the two identifiers as well: the **checkpoint ID** (a `UserMessage.uuid`, saying *which moment*) and the **session ID** (a `ResultMessage.session_id`, saying *which session*).

The final cell prints the state of all four files after both demos. Expect `catalog.txt` to match its original, `summary.txt` and `parking.txt` to be missing, and `notes.txt` to differ from its original because of the Bash line that survived.

In [15]:
# Cell 15: Final file states after both demos
print("File states after both demos")
print("-" * 44)

for name in ["catalog.txt", "notes.txt", "summary.txt", "parking.txt"]:
    path = DEMO_DIR / name
    if not path.exists():
        print(f"{name:<14} missing")
    elif name in ORIGINAL_HASHES:
        same = sha256_of(path) == ORIGINAL_HASHES[name]
        print(f"{name:<14} exists, matches original: {same}")
    else:
        print(f"{name:<14} exists")

File states after both demos
--------------------------------------------
catalog.txt    exists, matches original: True
notes.txt      exists, matches original: False
summary.txt    missing
parking.txt    missing
